# Titanic Survival Classifier

A Scikit-learn pipeline predicting passenger survival on the Titanic dataset. Includes a baseline Random Forest model and a tuned Gradient Boosting model with feature engineering.

## Baseline Model

In [ ]:
# --- Baseline Model: Random Forest ---
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

df = pd.read_csv("train.csv")

df["Age"] = SimpleImputer(strategy="median").fit_transform(df[["Age"]])
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df["Sex"] = LabelEncoder().fit_transform(df["Sex"])
df["Embarked"] = LabelEncoder().fit_transform(df["Embarked"])

features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
X = df[features]
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
preds = model.predict(X_test)

print("Baseline Accuracy:", accuracy_score(y_test, preds))
print("Baseline F1-Score:", f1_score(y_test, preds))

## Tuned Model (Feature Engineering + Gradient Boosting + GridSearchCV)

In [ ]:
# --- Tuned Model: Gradient Boosting + Feature Engineering + GridSearchCV ---
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier

df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
df["Title"] = df["Title"].replace(
    ["Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major", "Rev", "Sir", "Jonkheer", "Dona"],
    "Rare"
)
df["Title"] = df["Title"].replace(["Mlle", "Ms"], "Miss")
df["Title"] = df["Title"].replace("Mme", "Mrs")

df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["Fare"] = SimpleImputer(strategy="median").fit_transform(df[["Fare"]])

title_dummies = pd.get_dummies(df["Title"], prefix="Title")
df = pd.concat([df, title_dummies], axis=1)

base_features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked", "FamilySize"]
features = base_features + list(title_dummies.columns)

X = df[features]
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [2, 3, 4],
    "learning_rate": [0.01, 0.05, 0.1]
}

grid_search = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)

best_model = grid_search.best_estimator_
preds = best_model.predict(X_test)

print("Tuned Accuracy:", accuracy_score(y_test, preds))
print("Tuned F1-Score:", f1_score(y_test, preds))